# Data Preprocessing

This notebook handles data preprocessing including:
- Loading DICOM images
- Converting to grayscale
- Resizing images
- Normalization
- Train/Validation/Test split

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

from src.data.data_loader import load_dicom_images, load_labels_from_csv, create_label_mapping
from src.data.preprocessing import preprocess_images, split_data
from src.utils.visualization import plot_images
from src.utils.config import CLASS_LABELS, IMG_SIZE

plt.style.use('seaborn-v0_8')

## 1. Load Images and Labels

In [ ]:
# Load data
pneumonia_files_path = os.path.join(os.path.expanduser('~'), 'Desktop', 'pneumonia_files')
train_zip = os.path.join(pneumonia_files_path, 'stage_2_train_images.zip')
train_labels_path = os.path.join(pneumonia_files_path, 'stage_2_train_labels.csv')
class_info_path = os.path.join(pneumonia_files_path, 'stage_2_detailed_class_info.csv')

# Load images and labels
print("Loading images...")
images, labels = load_dicom_images(
    zip_path=train_zip,
    labels_csv=train_labels_path,
    class_info_csv=class_info_path
)

print(f"Loaded {len(images)} images")
print(f"Image shape (before preprocessing): {images[0].shape}")
print(f"Labels shape: {labels.shape}")
print(f"Unique labels: {np.unique(labels)}")

## 2. Visualize Images Before Preprocessing

In [ ]:
# Plot sample images before preprocessing
sample_indices = np.random.choice(len(images), min(9, len(images)), replace=False)
plot_images(images[sample_indices], 
           labels[sample_indices],
           class_names=list(CLASS_LABELS.values()),
           num_images=9)

## 3. Preprocess Images (Grayscale, Resize, Normalize)

In [ ]:
# Preprocess images
print("Preprocessing images...")
processed_images = preprocess_images(
    images,
    grayscale=True,
    resize=True,
    target_size=IMG_SIZE,
    normalize=True,
    normalization_method='min_max'
)

print(f"Processed images shape: {processed_images.shape}")
print(f"Image value range: [{processed_images.min():.3f}, {processed_images.max():.3f}]")

## 4. Visualize Images After Preprocessing

In [ ]:
# Plot sample images after preprocessing
plot_images(processed_images[sample_indices], 
           labels[sample_indices],
           class_names=list(CLASS_LABELS.values()),
           num_images=9)

## 5. Split Data into Train, Validation, and Test Sets

In [ ]:
# Split data
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    processed_images,
    labels,
    test_size=0.15,
    val_size=0.15,
    random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Check class distribution in each set
print("\nTraining set class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {CLASS_LABELS[u]}: {c} ({c/len(y_train)*100:.1f}%)")

print("\nValidation set class distribution:")
unique, counts = np.unique(y_val, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {CLASS_LABELS[u]}: {c} ({c/len(y_val)*100:.1f}%)")

print("\nTest set class distribution:")
unique, counts = np.unique(y_test, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {CLASS_LABELS[u]}: {c} ({c/len(y_test)*100:.1f}%)")

## 6. Save Preprocessed Data (Optional)

You can save the preprocessed data for later use to avoid reprocessing.

In [ ]:
# Save preprocessed data (uncomment to save)
# import pickle
# 
# data_dict = {
#     'X_train': X_train,
#     'X_val': X_val,
#     'X_test': X_test,
#     'y_train': y_train,
#     'y_val': y_val,
#     'y_test': y_test
# }
# 
# with open('data/processed/preprocessed_data.pkl', 'wb') as f:
#     pickle.dump(data_dict, f)
# 
# print("Preprocessed data saved successfully!")